# Task 8: Feature Engineering and Model Tuning

## Section 1: Feature Engineering & Model Tuning

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
df = pd.read_csv("student-scores.csv")
print(df.head())

   id first_name last_name                                  email  gender  \
0   1       Paul     Casey         paul.casey.1@gslingacademy.com    male   
1   2   Danielle  Sandoval  danielle.sandoval.2@gslingacademy.com  female   
2   3       Tina   Andrews       tina.andrews.3@gslingacademy.com  female   
3   4       Tara     Clark         tara.clark.4@gslingacademy.com  female   
4   5    Anthony    Campos     anthony.campos.5@gslingacademy.com    male   

   part_time_job  absence_days  extracurricular_activities  \
0          False             3                       False   
1          False             2                       False   
2          False             9                        True   
3          False             5                       False   
4          False             5                       False   

   weekly_self_study_hours   career_aspiration  math_score  history_score  \
0                       27              Lawyer          73             81   
1         

#### Student performance is evaluated by computing their total score as the sum of individual subject scores. A passing status is then determined by checking if the total score meets or exceeds 60% of the maximum possible score, assigning 1 for pass and 0 for fail.

In [4]:
subject_columns = ['math_score', 'history_score', 'physics_score', 'chemistry_score', 
                   'biology_score', 'english_score', 'geography_score']
df['total_score'] = df[subject_columns].sum(axis=1)
passing_threshold = 0.6 * (100 * len(subject_columns))  
df['Passed'] = (df['total_score'] >= passing_threshold).astype(int)

In [5]:
X = df[subject_columns + ['total_score']]  
y = df['Passed'] 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### A RandomForestClassifier is set up with a fixed random state, and a grid of hyperparameters is defined for tuning, including tree count, depth, and split criteria.

In [6]:
model = RandomForestClassifier(random_state=42)
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

#### GridSearchCV performs hyperparameter tuning on the model using cross-validation, optimizing for accuracy. It then fits the best combination to the training data and prints the optimal parameters.

In [8]:
import warnings
warnings.filterwarnings("ignore")

grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)

Fitting 5 folds for each of 81 candidates, totalling 405 fits
Best Parameters: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}


In [9]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.9975


## Section 2: Fraud Detection with Decision Trees

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

In [13]:
df = pd.read_csv("Fraud.csv")
print(df.head())

   step      type    amount     nameOrig  oldbalanceOrg  newbalanceOrig  \
0     1   PAYMENT   9839.64  C1231006815       170136.0       160296.36   
1     1   PAYMENT   1864.28  C1666544295        21249.0        19384.72   
2     1  TRANSFER    181.00  C1305486145          181.0            0.00   
3     1  CASH_OUT    181.00   C840083671          181.0            0.00   
4     1   PAYMENT  11668.14  C2048537720        41554.0        29885.86   

      nameDest  oldbalanceDest  newbalanceDest  isFraud  isFlaggedFraud  
0  M1979787155             0.0             0.0        0               0  
1  M2044282225             0.0             0.0        0               0  
2   C553264065             0.0             0.0        1               0  
3    C38997010         21182.0             0.0        1               0  
4  M1230701703             0.0             0.0        0               0  


#### Missing values are filled with the median of numeric columns, while categorical variables are label-encoded to convert them into numerical form for model compatibility.

In [21]:
df.fillna(df.median(numeric_only=True), inplace=True)
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    df[col] = LabelEncoder().fit_transform(df[col])
df["amount_per_category"] = df["amount"] / (df["type"] + 1) 

In [22]:
X = df.drop(columns=['isFraud']) 
y = df['isFraud']

#### The dataset is split into training and testing sets, with 20% reserved for testing. A DecisionTreeClassifier is trained on the training data and then used to predict outcomes on the test set.

In [23]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

In [24]:
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9996089931573803
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    209491
           1       0.81      0.83      0.82       224

    accuracy                           1.00    209715
   macro avg       0.91      0.91      0.91    209715
weighted avg       1.00      1.00      1.00    209715

